## Step 0: Mount Google Drive
This cell mounts Google Drive to the `/content/drive` directory, allowing persistent access to datasets and model checkpoints. We define the `PROJECT_ROOT` to point to our dedicated project folder.

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the project root directory in your Drive
# Please make sure you have a folder named 'ML2_Project' in your Google Drive
PROJECT_ROOT = '/content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi'

if not os.path.exists(PROJECT_ROOT):
    os.makedirs(PROJECT_ROOT)
    print(f"Created project folder at: {PROJECT_ROOT}")
else:
    print(f"Project folder found: {PROJECT_ROOT}")

Mounted at /content/drive
Project folder found: /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi


## Cell 1: Environment Setup and Global Configuration
This section initializes the project environment by defining the directory paths for raw data, processed images, and model baselines. It also configures global hyperparameters (IMG_SIZE, SEED) and verifies the availability of GPU (CUDA) for accelerated feature extraction. We set a manual seed to ensure results are reproducible across different runs.

In [21]:
import os
import random
import numpy as np
import pandas as pd
import torch

# --- Global Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Path Definitions (Google Drive Based) ---
# Note: Ensure your CSV is located at 'ML2_Project/data/raw/data.csv'
DATA_DIR = os.path.join(PROJECT_ROOT, "data/raw/images")
CSV_PATH = os.path.join(PROJECT_ROOT, "data/raw/data.csv")
PROCESSED_PATH = os.path.join(PROJECT_ROOT, "data/processed")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "models/ryan/baselines")

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Model & Device Configuration ---
IMG_SIZE = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"--- Project Configuration Loaded ---")
print(f"Execution Device: {DEVICE}")
print(f"CSV Metadata Path: {CSV_PATH}")
print(f"Images Path: {DATA_DIR}")

# Sanity Check for Files
if os.path.exists(CSV_PATH):
    print("SUCCESS: Found data.csv")
else:
    print("WARNING: data.csv not found. Please check your Drive path.")

--- Project Configuration Loaded ---
Execution Device: cuda
CSV Metadata Path: /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi/data/raw/data.csv
Images Path: /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi/data/raw/images
SUCCESS: Found data.csv


## Step 1.5: Data Integrity Check and Automatic Cleaning
Before instantiating the dataset, we must ensure physical parity between the metadata (CSV) and the storage (Google Drive). This cell scans the `DATA_DIR` for every image name listed in the raw metadata. If missing files are detected, it automatically filters the entries and generates a `clean_data.csv` within the `processed` directory. This proactive cleaning prevents the injection of "blank tensor" noise into the feature extraction process, ensuring that the ML Researcher (Role 4) receives only high-quality, valid data points.

In [22]:
# Cell 1.5: Data Integrity Check and Automatic Cleaning

# Define the path for the cleaned metadata in the processed folder
CLEAN_CSV_PATH = os.path.join(PROCESSED_PATH, "clean_data.csv")

print("🕵️ Starting Data Integrity Check...")

# Load the raw metadata
raw_metadata = pd.read_csv(CSV_PATH)
initial_count = len(raw_metadata)

# Perform vectorized check or loop to verify file existence
# Note: os.path.exists can be slow on Drive for huge datasets;
# for 4k images, this loop is acceptable.
valid_indices = []
for idx, row in raw_metadata.iterrows():
    img_path = os.path.join(DATA_DIR, row['name'])
    if os.path.exists(img_path):
        valid_indices.append(idx)

# Filter the dataframe
clean_metadata = raw_metadata.iloc[valid_indices].reset_index(drop=True)
missing_count = initial_count - len(clean_metadata)

# Save the cleaned CSV to the PROCESSED directory
clean_metadata.to_csv(CLEAN_CSV_PATH, index=False)

print(f"--- Integrity Report ---")
print(f"Total entries in raw CSV:  {initial_count}")
print(f"Missing image files:       {missing_count}")
print(f"Valid entries preserved:   {len(clean_metadata)}")
print(f"✅ Cleaned metadata saved to: {CLEAN_CSV_PATH}")

# Update CSV_PATH variable for subsequent cells to ensure the pipeline uses clean data
CURRENT_CSV_PATH = CLEAN_CSV_PATH

🕵️ Starting Data Integrity Check...
--- Integrity Report ---
Total entries in raw CSV:  4206
Missing image files:       244
Valid entries preserved:   3962
✅ Cleaned metadata saved to: /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi/data/processed/clean_data.csv


## Cell 2: Dataset and DataLoader Implementation
In this section, we implement a custom PyTorch `Dataset` class (`FaceBMIDataset`) to handle the loading of face images and their associated metadata. This class maps image filenames to their respective labels and filters the dataset into training and testing sets based on the `is_training` flag.

We define a transformation pipeline (`torchvision.transforms`) that resizes the images to 224x224 and applies ImageNet normalization (mean and std), which is strictly required by the pre-trained VGG-Face architecture. Finally, `DataLoaders` are instantiated to manage batch processing.

In [23]:
# Cell 2: Dataset and DataLoader Implementation

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import torch
import os

class FaceBMIDataset(Dataset):
    def __init__(self, csv_file, root_dir, is_train=True, transform=None):
        """
        Custom Dataset for Face-to-BMI extraction.
        Args:
            csv_file (string): Path to the csv file with annotations.
            root_dir (string): Directory with all the images.
            is_train (bool): Filter dataset for training (1) or testing (0).
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        # Load the entire metadata CSV
        self.metadata = pd.read_csv(csv_file)

        # Filter based on train/test split using the 'is_training' column
        target_flag = 1 if is_train else 0
        self.metadata = self.metadata[self.metadata['is_training'] == target_flag].reset_index(drop=True)

        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        # Construct the full image path
        img_name = os.path.join(self.root_dir, self.metadata.loc[idx, 'name'])

        # Open image and convert to RGB (VGG expects 3 channels)
        try:
            image = Image.open(img_name).convert('RGB')
        except FileNotFoundError:
            # Safe fallback if an image is missing from the Drive folder
            print(f"Warning: Image {img_name} not found. Returning a blank tensor.")
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        # Extract the BMI target value
        bmi = float(self.metadata.loc[idx, 'bmi'])

        # Apply VGG-Face specific transformations
        if self.transform:
            image = self.transform(image)

        # Return a dictionary containing the image tensor, the BMI value, and the filename
        sample = {
            'image': image,
            'bmi': torch.tensor(bmi, dtype=torch.float32),
            'filename': self.metadata.loc[idx, 'name']
        }
        return sample

# --- Transformation Pipeline ---
# Standard ImageNet normalization is required for weights pre-trained on VGG architectures
vgg_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- Instantiate Datasets ---
# DATA_DIR and CSV_PATH are inherited from Cell 1
train_dataset = FaceBMIDataset(csv_file=CURRENT_CSV_PATH, root_dir=DATA_DIR, is_train=True, transform=vgg_transform)
test_dataset = FaceBMIDataset(csv_file=CURRENT_CSV_PATH, root_dir=DATA_DIR, is_train=False, transform=vgg_transform)

# --- Instantiate DataLoaders ---
# We use drop_last=False for extraction to ensure we don't miss any images at the end of the batch
# num_workers=2 is optimal for Colab's standard environment
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ Train dataset loaded with {len(train_dataset)} samples.")
print(f"✅ Test dataset loaded with {len(test_dataset)} samples.")
print(f"✅ DataLoaders initialized (Batch Size: {BATCH_SIZE}). Ready for feature extraction.")

✅ Train dataset loaded with 3210 samples.
✅ Test dataset loaded with 752 samples.
✅ DataLoaders initialized (Batch Size: 32). Ready for feature extraction.


## Cell 3: VGG-Face Model Architecture and Weight Loading
This cell focuses on the initialization of the VGG-Face model. We load the VGG16-based architecture, which serves as the foundation for VGG-Face. Since our primary goal for Week 1 is feature extraction (acting as a baseline for the regression task), we modify the network to truncate the final classification layers.

Our target is the **fc6** layer (the first fully connected layer combined with its ReLU activation), which outputs a 4096-dimensional representation. We explicitly set the model to evaluation mode (`eval()`) and freeze all gradients (`requires_grad = False`) to treat it as a fixed feature extractor, significantly reducing GPU memory footprint during inference.

In [24]:
# Cell 3: VGG-Face Model Architecture and Truncation

import torchvision.models as models
import torch.nn as nn

print("Loading VGG16 architecture (base for VGG-Face)...")

# Load the standard VGG16 architecture
# Note: For a strict 100% replication of the 2017 paper, one would load the specific
# Oxford VGG-Face .pth weights here. For this baseline step, we use the standard
# ImageNet pre-trained weights to validate the pipeline architecture.
vgg_model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# --- Truncate the Network ---
# The original VGG16 classifier consists of:
# (0): Linear(in_features=25088, out_features=4096) -> This is fc6
# (1): ReLU(inplace=True)
# (2): Dropout(p=0.5)
# (3): Linear(in_features=4096, out_features=4096) -> This is fc7
# ... and so on.
# We truncate the classifier to only keep up to fc6 and its ReLU activation (indices 0 and 1).

truncated_classifier = nn.Sequential(*list(vgg_model.classifier.children())[:2])
vgg_model.classifier = truncated_classifier

# --- Freeze the Weights ---
# We are using this strictly as a feature extractor, so we do not need gradients.
# This prevents weights from updating and saves memory.
for param in vgg_model.parameters():
    param.requires_grad = False

# Move the model to GPU (if available) and set to evaluation mode
vgg_model = vgg_model.to(DEVICE)
vgg_model.eval()

print("✅ Model successfully loaded, truncated, and frozen.")
print("--- Current Classifier Architecture ---")
print(vgg_model.classifier)
print(f"✅ Target output dimension: {vgg_model.classifier[0].out_features}")

Loading VGG16 architecture (base for VGG-Face)...
✅ Model successfully loaded, truncated, and frozen.
--- Current Classifier Architecture ---
Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
)
✅ Target output dimension: 4096


## Cell 4: Feature Extraction Execution
This is the core execution block where we iterate through the `DataLoader` to pass batches of face images through the truncated VGG-Face network. The resulting 4096-dimensional vectors are collected and aggregated into a high-dimensional feature matrix.

We separate the extraction process for the training and testing sets based on the `is_training` flag provided in the metadata, strictly adhering to the experimental setup of the original paper to prevent data leakage. The final matrices and their corresponding BMI labels are serialized and saved as `.npy` files for the regression stage.

In [25]:
# Cell 4: Feature Extraction Loop
from tqdm.auto import tqdm

def extract_features(data_loader, model, device):
    """
    Passes images through the model and collects the output features.
    """
    features_list = []
    bmi_list = []
    filenames_list = []

    # Disable gradient calculation for faster inference and reduced memory usage
    with torch.no_grad():
        # tqdm adds a nice progress bar
        for batch in tqdm(data_loader, desc="Extracting"):
            images = batch['image'].to(device)
            bmis = batch['bmi'].numpy() # Keep labels on CPU
            filenames = batch['filename']

            # Forward pass through the truncated VGG model
            # Output shape should be (Batch_Size, 4096)
            batch_features = model(images)

            # Move features back to CPU and convert to numpy array
            features_list.append(batch_features.cpu().numpy())
            bmi_list.extend(bmis)
            filenames_list.extend(filenames)

    # Concatenate all batches into single massive numpy arrays
    final_features = np.vstack(features_list)
    final_bmis = np.array(bmi_list)
    final_filenames = np.array(filenames_list)

    return final_features, final_bmis, final_filenames

print("🚀 Starting Feature Extraction for Training Set...")
train_features, train_bmis, train_filenames = extract_features(train_loader, vgg_model, DEVICE)

print("🚀 Starting Feature Extraction for Testing Set...")
test_features, test_bmis, test_filenames = extract_features(test_loader, vgg_model, DEVICE)

# --- Save to Disk ---
print(f"💾 Saving extracted features to: {OUTPUT_DIR}")

np.save(os.path.join(OUTPUT_DIR, 'train_features.npy'), train_features)
np.save(os.path.join(OUTPUT_DIR, 'train_bmis.npy'), train_bmis)
np.save(os.path.join(OUTPUT_DIR, 'train_filenames.npy'), train_filenames)

np.save(os.path.join(OUTPUT_DIR, 'test_features.npy'), test_features)
np.save(os.path.join(OUTPUT_DIR, 'test_bmis.npy'), test_bmis)
np.save(os.path.join(OUTPUT_DIR, 'test_filenames.npy'), test_filenames)

print("✅ Feature extraction and saving completed successfully!")

🚀 Starting Feature Extraction for Training Set...


Extracting:   0%|          | 0/101 [00:00<?, ?it/s]

🚀 Starting Feature Extraction for Testing Set...


Extracting:   0%|          | 0/24 [00:00<?, ?it/s]

💾 Saving extracted features to: /content/drive/MyDrive/ML2-Team2/Final Project/face-to-bmi/models/ryan/baselines
✅ Feature extraction and saving completed successfully!


## Cell 5: Sanity Check & Validation
To guarantee the integrity of our data pipeline, we perform a final sanity check before handing the data over to the regression task (Role 4). This cell reloads the newly created `.npy` files from the disk and verifies their dimensional shapes.

We expect the training feature matrix to be $(3368, 4096)$ and the testing feature matrix to be $(838, 4096)$. Additionally, we perform a programmatic check for any `NaN` (Not a Number) values or extreme anomalies to ensure the features are mathematically sound for the SVR model.

In [27]:
# Cell 5: Sanity Check & Validation

print("🔍 Loading extracted features from disk for validation...")
# Load the saved numpy arrays to ensure they were written correctly
val_train_features = np.load(os.path.join(OUTPUT_DIR, 'train_features.npy'))
val_train_bmis = np.load(os.path.join(OUTPUT_DIR, 'train_bmis.npy'))
val_train_filenames = np.load(os.path.join(OUTPUT_DIR, 'train_filenames.npy'))

val_test_features = np.load(os.path.join(OUTPUT_DIR, 'test_features.npy'))
val_test_bmis = np.load(os.path.join(OUTPUT_DIR, 'test_bmis.npy'))
val_test_filenames = np.load(os.path.join(OUTPUT_DIR, 'test_filenames.npy'))

print("\n--- 📏 Dimensionality (Shape) Validation ---")
# The shapes should exactly match our dataset count and the VGG fc6 dimension (4096)
print(f"Train Features Shape: {val_train_features.shape}  | Expected: (3210, 4096)")
print(f"Train BMIs Shape:     {val_train_bmis.shape}        | Expected: (3368,)")
print(f"Test Features Shape:  {val_test_features.shape}   | Expected: (752, 4096)")

print("\n--- 🧮 Data Integrity Check ---")
# Check for NaNs (which would crash the SVR regressor)
has_nan_train = np.isnan(val_train_features).any()
has_nan_test = np.isnan(val_test_features).any()
print(f"Contains NaN in Train Features? {'❌ YES' if has_nan_train else '✅ NO'}")
print(f"Contains NaN in Test Features?  {'❌ YES' if has_nan_test else '✅ NO'}")

# Quick preview of the first feature vector's head
print("\n--- 🔬 Feature Preview (First 5 values of Train Image 0) ---")
print(val_train_features[0][:5])

print("\n🎉 Week 1 Role 3 Mission Accomplished! Data is ready for Role 4.")

🔍 Loading extracted features from disk for validation...

--- 📏 Dimensionality (Shape) Validation ---
Train Features Shape: (3210, 4096)  | Expected: (3210, 4096)
Train BMIs Shape:     (3210,)        | Expected: (3368,)
Test Features Shape:  (752, 4096)   | Expected: (752, 4096)

--- 🧮 Data Integrity Check ---
Contains NaN in Train Features? ✅ NO
Contains NaN in Test Features?  ✅ NO

--- 🔬 Feature Preview (First 5 values of Train Image 0) ---
[0.        0.        2.2817948 0.        0.       ]

🎉 Week 1 Role 3 Mission Accomplished! Data is ready for Role 4.
